In [1]:
# Parameters
base_path = "C:\\TCC_USP"
run_id = "20251126_165642"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg
from src.validation import merges
from src.utils.logger import log_result

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "20_final_dashboard_analysis"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\TCC_USP
PROC_PATH: C:\TCC_USP\data_processed


In [3]:
# 2. Bibliotecas e verificação de arquivos
import json
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

paths = {
    "labels": os.path.join(PROC_PATH, "labels_y_daily.csv"),
    "oof": os.path.join(PROC_PATH, "16_oof_predictions.csv"),
    "results16": os.path.join(PROC_PATH, "results_16_models_tfidf.json"),
    "corr": os.path.join(PROC_PATH, "17_sentiment_correlations.csv"),
    "anova": os.path.join(PROC_PATH, "17_sentiment_anova.csv"),
    "lags": os.path.join(PROC_PATH, "17_sentiment_lags.csv"),
    "sentiment_html": os.path.join(PROC_PATH, "17_sentiment_dashboard.html"),
    "perf": os.path.join(PROC_PATH, "18_backtest_results.csv"),
    "daily": os.path.join(PROC_PATH, "18_backtest_daily_curves.csv"),
}

required = ["labels", "oof", "results16"]
for key in required:
    if not os.path.exists(paths[key]):
        raise FileNotFoundError(f"Arquivo obrigatório ausente: {paths[key]}")

print(json.dumps(paths, indent=2, ensure_ascii=False))

{
  "labels": "C:\\TCC_USP\\data_processed\\labels_y_daily.csv",
  "oof": "C:\\TCC_USP\\data_processed\\16_oof_predictions.csv",
  "results16": "C:\\TCC_USP\\data_processed\\results_16_models_tfidf.json",
  "corr": "C:\\TCC_USP\\data_processed\\17_sentiment_correlations.csv",
  "anova": "C:\\TCC_USP\\data_processed\\17_sentiment_anova.csv",
  "lags": "C:\\TCC_USP\\data_processed\\17_sentiment_lags.csv",
  "sentiment_html": "C:\\TCC_USP\\data_processed\\17_sentiment_dashboard.html",
  "perf": "C:\\TCC_USP\\data_processed\\18_backtest_results.csv",
  "daily": "C:\\TCC_USP\\data_processed\\18_backtest_daily_curves.csv"
}


In [4]:
# 3. Carregar datasets
labels = pd.read_csv(paths["labels"], parse_dates=["day"])
oof = pd.read_csv(paths["oof"], parse_dates=["day"])
with open(paths["results16"], "r", encoding="utf-8") as f:
    results16 = json.load(f)

corr_df = pd.read_csv(paths["corr"]) if os.path.exists(paths["corr"]) else pd.DataFrame()
anova_df = pd.read_csv(paths["anova"]) if os.path.exists(paths["anova"]) else pd.DataFrame()
lag_df = pd.read_csv(paths["lags"]) if os.path.exists(paths["lags"]) else pd.DataFrame()
perf_df = pd.read_csv(paths["perf"]) if os.path.exists(paths["perf"]) else pd.DataFrame()
daily_df = pd.read_csv(paths["daily"], parse_dates=["day"]) if os.path.exists(paths["daily"]) else pd.DataFrame()

display(oof.head())
print("Perf rows:", len(perf_df))

,model,fold,row_id,day,y,ret_next,close,proba
0,logreg_l2,0,390,2019-08-05,1,0.020640,100098.0,0.496225
1,logreg_l2,0,391,2019-08-06,1,0.006049,102164.0,0.487361
2,logreg_l2,0,392,2019-08-07,1,0.012969,102782.0,0.510332
3,logreg_l2,0,393,2019-08-08,0,-0.001143,104115.0,0.464429
4,logreg_l2,0,394,2019-08-09,0,-0.020010,103996.0,0.478652


Perf rows: 6


In [5]:
merges.check_intersection(oof, labels, col_left="day", col_right="day", min_days=90)

[day] 2019-08-05 → 2025-11-17 | 1552 dias únicos | 3104 registros | missing=0
[day] 2018-01-02 → 2025-11-19 | 2771 dias únicos | 2771 registros | missing=0
Dias em comum: 1552


{'left': DateSummary(min_date=Timestamp('2019-08-05 00:00:00'), max_date=Timestamp('2025-11-17 00:00:00'), n_days=1552, n_records=3104, n_missing=0),
 'right': DateSummary(min_date=Timestamp('2018-01-02 00:00:00'), max_date=Timestamp('2025-11-19 00:00:00'), n_days=2771, n_records=2771, n_missing=0),
 'common_days': 1552,
 'days_list': DatetimeIndex(['2019-08-05', '2019-08-06', '2019-08-07', '2019-08-08',
                '2019-08-09', '2019-08-12', '2019-08-13', '2019-08-14',
                '2019-08-15', '2019-08-16',
                ...
                '2025-11-04', '2025-11-05', '2025-11-06', '2025-11-07',
                '2025-11-10', '2025-11-11', '2025-11-12', '2025-11-13',
                '2025-11-14', '2025-11-17'],
               dtype='datetime64[ns]', length=1552, freq=None)}

In [6]:
# 4. Consolidar métricas de modelos
model_rows = []
for model_name, metrics in results16.get("models", {}).items():
    model_rows.append({
        "model": model_name,
        "auc": metrics["auc"].get("value"),
        "mda": metrics["mda"].get("value"),
    })
metrics_df = pd.DataFrame(model_rows)

if not corr_df.empty:
    metrics_df = metrics_df.merge(corr_df[["model", "pearson_r", "spearman_r"]], on="model", how="left")
if not perf_df.empty:
    best_perf = perf_df.sort_values("sharpe", ascending=False).groupby("model").head(1)
    metrics_df = metrics_df.merge(
        best_perf[["model", "strategy", "cagr", "sharpe", "max_dd", "hit_ratio"]],
        on="model",
        how="left",
    )

metrics_df.sort_values("auc", ascending=False, inplace=True)
display(metrics_df)

,model,auc,mda,pearson_r,spearman_r,strategy,cagr,sharpe,max_dd,hit_ratio
0,logreg_l2,0.501538,0.502577,-0.673632,-0.724207,long_only_60,0.008904,0.741344,-0.002482,0.002577
1,rf_200,0.491264,0.511598,0.008709,-0.048365,long_only_55,-0.032055,-0.132767,-0.345637,0.057345


In [7]:
# 5. Construir dataset diário para gráficos
merged = oof.merge(labels[["day", "ret_next"]], on="day", how="left")
merged.sort_values(["model", "day"], inplace=True)
display(merged.head())

,model,fold,row_id,day,y,ret_next_x,close,proba,ret_next_y
0,logreg_l2,0,390,2019-08-05,1,0.020640,100098.0,0.496225,0.020640
1,logreg_l2,0,391,2019-08-06,1,0.006049,102164.0,0.487361,0.006049
2,logreg_l2,0,392,2019-08-07,1,0.012969,102782.0,0.510332,0.012969
3,logreg_l2,0,393,2019-08-08,0,-0.001143,104115.0,0.464429,-0.001143
4,logreg_l2,0,394,2019-08-09,0,-0.020010,103996.0,0.478652,-0.020010


In [8]:
# 6. Figura principal: retornos vs probabilidades
main_fig = make_subplots(specs=[[{"secondary_y": True}]])
main_fig.add_trace(
    go.Bar(
        x=labels["day"],
        y=labels["ret_next"],
        name="Retorno Ibov D+1",
        marker_color="rgba(55,83,109,0.6)",
    ),
    secondary_y=False,
)
for model_name, sub in merged.groupby("model"):
    main_fig.add_trace(
        go.Scatter(
            x=sub["day"],
            y=sub["proba"],
            mode="lines+markers",
            name=f"P(up) {model_name}",
        ),
        secondary_y=True,
    )
main_fig.update_layout(title="Sentimento diário vs retorno futuro", hovermode="x unified")
main_fig.update_yaxes(title_text="Retorno Ibov", secondary_y=False)
main_fig.update_yaxes(title_text="Probabilidade de alta", range=[0, 1], secondary_y=True)
main_fig.show()

In [9]:
# 7. Figura de equity das estratégias (notebook 18)
if not daily_df.empty:
    equity_fig = go.Figure()
    for (model_name, strat_name), sub in daily_df.groupby(["model", "strategy"]):
        equity_fig.add_trace(
            go.Scatter(
                x=sub["day"],
                y=sub["equity"],
                mode="lines",
                name=f"{model_name} | {strat_name}",
            )
        )
    equity_fig.update_layout(title="Curvas de capital simuladas", yaxis_title="Equity")
    equity_fig.show()
else:
    equity_fig = None

In [10]:
# 8. Tabela final consolidada
summary_df = metrics_df.copy()
if not anova_df.empty:
    summary_df = summary_df.merge(
        anova_df[["model", "p_value", "mean_neg", "mean_pos"]],
        on="model",
        how="left",
        suffixes=("", "_anova"),
    )
summary_df.rename(columns={"p_value": "anova_p", "mean_neg": "ret_neg", "mean_pos": "ret_pos"}, inplace=True)
display(summary_df)

,model,auc,mda,pearson_r,spearman_r,strategy,cagr,sharpe,max_dd,hit_ratio,anova_p,ret_neg,ret_pos
0,logreg_l2,0.501538,0.502577,-0.673632,-0.724207,long_only_60,0.008904,0.741344,-0.002482,0.002577,0.002611,0.005,-0.005000
1,rf_200,0.491264,0.511598,0.008709,-0.048365,long_only_55,-0.032055,-0.132767,-0.345637,0.057345,0.773204,0.000,-0.001667


In [11]:
top_model = summary_df.sort_values(["auc", "mda"], ascending=False).iloc[0]
metrics = {"auc": float(top_model.get("auc", 0) or 0), "mda": float(top_model.get("mda", 0) or 0)}
extra = {
    "dataset": "dashboard_consolidado",
    "model": top_model.get("model"),
    "best_sharpe": float(top_model.get("sharpe", 0) or 0),
}
log_result(model_name="dashboard_summary", dataset_name="dashboard_consolidado", metrics=metrics, extra=extra)


C:\TCC_USP\tcc-usp-ibovespa-sentimento\venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning:

Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534



{'timestamp': '2025-11-26T19:56:54.040887',
 'model': 'dashboard_summary',
 'dataset': 'dashboard_consolidado',
 'metrics': {'auc': 0.5015378221113882, 'mda': 0.5025773195876289},
 'extra': {'dataset': 'dashboard_consolidado',
  'model': 'logreg_l2',
  'best_sharpe': 0.741343669693607}}

In [12]:
# 9. Gerar dashboard HTML final
html_file = os.path.join(PROC_PATH, "20_final_dashboard.html")
with open(html_file, "w", encoding="utf-8") as f:
    f.write("<h1>Dashboard Final - Sentimento x Ibovespa</h1>\n")
    f.write(f"<p>Atualizado em {STAMP}</p>")
    f.write("<h2>Metricas por modelo</h2>")
    f.write(summary_df.to_html(index=False, float_format=lambda x: f"{x:0.3f}" if isinstance(x, (int, float)) else x))
    f.write("<h2>Retornos vs Probabilidades</h2>")
    f.write(main_fig.to_html(full_html=False, include_plotlyjs="cdn"))
    if equity_fig is not None:
        f.write("<h2>Backtests</h2>")
        f.write(equity_fig.to_html(full_html=False, include_plotlyjs=False))
    if os.path.exists(paths["sentiment_html"]):
        with open(paths["sentiment_html"], "r", encoding="utf-8") as sentiment_file:
            f.write("<h2>Analises de Sentimento (Notebook 17)</h2>")
            f.write(sentiment_file.read())

print("Dashboard salvo em:", html_file)


Dashboard salvo em: C:\TCC_USP\data_processed\20_final_dashboard.html


In [13]:
# 10. Resumo final
from IPython.display import Markdown

sections = [
    "20_final_dashboard_analysis concluido",
    f"Linhas de previsao: {len(merged)}",
    f"Modelos avaliados: {', '.join(sorted(merged['model'].unique()))}",
    "Artefatos atualizados:",
    "- 20_final_dashboard.html",
]
md = "\n".join(sections) + "\n\nPipeline completo!"
Markdown(md)


20_final_dashboard_analysis concluido
Linhas de previsao: 3104
Modelos avaliados: logreg_l2, rf_200
Artefatos atualizados:
- 20_final_dashboard.html

Pipeline completo!